In [1]:
import re
import torch
import pickle

from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader
from transformers.pipelines.pt_utils import KeyDataset
from transformers import pipeline,EsmModel, EsmTokenizer
from transformers import T5EncoderModel, T5Tokenizer

from torchinfo import summary
#from sklearn.decomposition import PCA
from pathlib import Path
from datasets import load_dataset,concatenate_datasets

outdir = Path("../data/encodings")
if not outdir.is_dir():
    outdir.mkdir()

In [2]:
### in progress , reestructurin the embedding calculation using functions

### 1. Downsampling non-cleavade samples matching number of positive cleavade samples , just for make visualization of embeddings easier

In [3]:
df_test = load_dataset("csv",data_files="../data/c_train.csv",split="train")
df_test.shape

(1135601, 2)

In [7]:
df_test["cleavage_window"]

['TRWEIIAEDE',
 'DFVRTLAEKR',
 'QPKQSWNPFP',
 'PVLVHLLSAI',
 'TIISSQRFYK',
 'ADNRIQELQQ',
 'GSRTSLQGER',
 'PSANTTLVSE',
 'AFPVLTHSTR',
 'EGPCGISYKT',
 'VYRAAGKEMI',
 'VFKILGAKSA',
 'RGWAAPLLLL',
 'TQALISYRDG',
 'EVESIPLPDM',
 'LILKDTEAVM',
 'HTGEKPYECK',
 'LKTLSELFSL',
 'RVGLRLLEVN',
 'QNEMPHIEAQ',
 'LPDHPELKKI',
 'EIKKRFRQLS',
 'PLRPIVVKDS',
 'PQALALPPLA',
 'IHNFHSSFES',
 'MKTLEVLTKN',
 'TSLGQERLTD',
 'EIIPKSAVGE',
 'GTGAMRHVVI',
 'THEHLAMTFD',
 'ILTALQIMYA',
 'KSRQTLQNAS',
 'AALIRSLPVE',
 'PEVGNPYSQQ',
 'EGKSAKERPI',
 'YIKFINLFPE',
 'DAMEYHTKIR',
 'AQFAARAIIS',
 'RKFPREGFGE',
 'HRPSELKKHV',
 'SGVGSSWRER',
 'DNPTPGIVIN',
 'AFAGDTRPRF',
 'QTLQANESAS',
 'IVIQKYHTVN',
 'QVEDLASLIS',
 'IPTGNSLYES',
 'PGQDFASGTL',
 'AEQVASFQEE',
 'LASELNKQYQ',
 'QKEHLLKYEY',
 'VEEEEEEESG',
 'ENGLLMTPCY',
 'DAFDQRGKRE',
 'GKSHHNWGKC',
 'LHLLSVLKHP',
 'LLLKELWLCV',
 'WQAHQFEAWI',
 'DPSSFSTKHS',
 'CELQFPQCLL',
 'AGAGAAKRKA',
 'LVSVLTTLER',
 'SELRSVLVMM',
 'IMMEMSDSET',
 'LYHLLSLNAV',
 'TTLFSHEKIH',
 'NLLYHVVG

In [12]:
df_test.filter(lambda x: x["elution_cleavage"] != 1).shape

Filter:   0%|          | 0/1135601 [00:00<?, ? examples/s]

(934716, 2)

In [3]:
positive_samples = df_test.filter(lambda x: x["elution_cleavage"] == 1)
negative_samples = df_test.filter(lambda x: x["elution_cleavage"] != 1).shuffle(seed=42)

downsampled_negative_samples = negative_samples.select(range(positive_samples.shape[0]))
downsampled_dataset = concatenate_datasets([positive_samples, downsampled_negative_samples])


In [2]:
downsampled_dataset.to_csv("data/c_test_downs_exp1.csv")

In [2]:
#df_train = load_dataset("csv",data_files="../data/c_train.csv",split="train")


### 2. Getting pLM embeddings on downsampled test data

In [4]:
# pLM chekcpoints , aka names of the model

ESMs = ["facebook/esm2_t33_650M_UR50D",
        "facebook/esm2_t36_3B_UR50D" ]

ProtT5 = [ "Rostlab/prot_t5_xl_half_uniref50-enc" ]

In [5]:
batch_size = 64

def load_model(model_name):
    
    if 'esm' in model_name:
        model_type = "esm"
        tokenizer = EsmTokenizer.from_pretrained(model_name)
        model = EsmModel.from_pretrained(model_name, device_map='auto',torch_dtype=torch.float16)
    else:
        model_type = "t5"
        tokenizer = T5Tokenizer.from_pretrained(model_name)
        model = T5EncoderModel.from_pretrained(model_name, device_map='auto',torch_dtype=torch.float16)
        
    return model, tokenizer, model_type

def embed_data(model_name, dataset, emb_type = None):

    if emb_type not in ["avg","concat"]:
        print("Computing per-residue embedding by default. Other emb_type = 'avg' or 'concat'" )
    
    model, tokenizer, model_type = load_model(model_name)
    pipe = pipeline("feature-extraction", model=model, tokenizer=tokenizer) 
    
    if model_type == "t5":
        dataset = dataset.map(process_batch)

    embeddings = []
    for out in tqdm(pipe(KeyDataset(dataset,'cleavage_window'),batch_size=batch_size, return_tensors=True,num_workers=10)):
        embeddings.append(out)
    embeddings = torch.cat(embeddings, dim=0)

    if emb_type == "avg":
        embeddings = embeddings[:,1:-1,:].mean(1)
    elif emb_type == "concat":
        embeddings = embeddings[:,1:-1,:].flatten(1)
    else:
        embeddings = embeddings[:,1:-1,:]
    
    embeddings = embeddings.numpy()

    emb_dict = {"label":dataset["elution_cleavage"],"embeddings":embeddings}

    # clean up gpu
    del model
    del tokenizer
    del dataset
    torch.cuda.empty_cache()

    
    return emb_dict
    #torch.save(embeddings,outdir/f"test_emb_{downsampled_dataset.shape[0]}seqs.pt")

def process_batch(batch):
    
    sequences = batch["cleavage_window"]
    seq = re.sub(r"[UZOB]", "X", sequences)
    batch_seq = ' '.join(list(seq))
    
    return {"cleavage_window": batch_seq}

In [4]:
downsampled_dataset = load_dataset("csv",data_files="../data/c_test_downs_exp1.csv", split="train")

In [5]:
downsampled_dataset["cleavage_window"][:10]

['TAFGGWKSVE',
 'AYKEAFEISK',
 'RLITRLENTQ',
 'QQEAALELEE',
 'FIMDRFPYYR',
 'DVRTTKCIAT',
 'VREAILEDLQ',
 'LAVTSYTDIA',
 'EKKRLKKEKK',
 'GHYTSFIRQH']

In [16]:
model, tokenizer, model_type = load_model(ESMs[0])
batch_encoding = tokenizer(downsampled_dataset["cleavage_window"], return_tensors="pt")

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
batch_encoding["input_ids"][:10]

tensor([[ 0, 11,  5, 18,  6,  6, 22, 15,  8,  7,  9,  2],
        [ 0,  5, 19, 15,  9,  5, 18,  9, 12,  8, 15,  2],
        [ 0, 10,  4, 12, 11, 10,  4,  9, 17, 11, 16,  2],
        [ 0, 16, 16,  9,  5,  5,  4,  9,  4,  9,  9,  2],
        [ 0, 18, 12, 20, 13, 10, 18, 14, 19, 19, 10,  2],
        [ 0, 13,  7, 10, 11, 11, 15, 23, 12,  5, 11,  2],
        [ 0,  7, 10,  9,  5, 12,  4,  9, 13,  4, 16,  2],
        [ 0,  4,  5,  7, 11,  8, 19, 11, 13, 12,  5,  2],
        [ 0,  9, 15, 15, 10,  4, 15, 15,  9, 15, 15,  2],
        [ 0,  6, 21, 19, 11,  8, 18, 12, 10, 16, 21,  2]])

In [22]:
summary(model,input_data=batch_encoding["input_ids"][:1],verbose=2)

Layer (type:depth-idx)                                  Output Shape              Param #
EsmModel                                                [1, 1280]                 661
├─EsmEmbeddings: 1-1                                    [1, 12, 1280]             1,313,280
│    └─word_embeddings.weight                                                     ├─42,240
│    └─position_embeddings.weight                                                 └─1,313,280
│    └─Embedding: 2-1                                   [1, 12, 1280]             42,240
│    │    └─weight                                                                └─42,240
├─EsmEncoder: 1-2                                       [1, 12, 1280]             --
│    └─layer.0.attention.self.query.weight                                        ├─1,638,400
│    └─layer.0.attention.self.query.bias                                          ├─1,280
│    └─layer.0.attention.self.key.weight                                          ├─1,638,400
│   

Layer (type:depth-idx)                                  Output Shape              Param #
EsmModel                                                [1, 1280]                 661
├─EsmEmbeddings: 1-1                                    [1, 12, 1280]             1,313,280
│    └─word_embeddings.weight                                                     ├─42,240
│    └─position_embeddings.weight                                                 └─1,313,280
│    └─Embedding: 2-1                                   [1, 12, 1280]             42,240
│    │    └─weight                                                                └─42,240
├─EsmEncoder: 1-2                                       [1, 12, 1280]             --
│    └─layer.0.attention.self.query.weight                                        ├─1,638,400
│    └─layer.0.attention.self.query.bias                                          ├─1,280
│    └─layer.0.attention.self.key.weight                                          ├─1,638,400
│   

### ESM-2 650M UR50D embeddings 

In [6]:
model_name = ESMs[0]

esm2_avg_dict = embed_data(model_name, downsampled_dataset, emb_type = "avg")
esm2_concat_dict = embed_data(model_name, downsampled_dataset, emb_type = "concat")
esm2_residue_dict = embed_data(model_name, downsampled_dataset)


NameError: name 'downsampled_dataset' is not defined

In [18]:
with open(outdir/"esm2_avg_test50540.pkl", "wb") as f:
    pickle.dump(esm2_avg_dict, f)

with open(outdir/"esm2_concat_test50540.pkl", "wb") as f:
    pickle.dump(esm2_concat_dict, f)

In [19]:
with open(outdir/"esm2_residue_test50540.pkl", "wb") as f:
    pickle.dump(esm2_residue_dict, f)

In [41]:
# test to see the similarity between vector embeddings.
#cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
#cos(esm2_embeddings_con[0].unsqueeze(0),esm2_concat[0].unsqueeze(0))


In [4]:
# asigning outputs to empty tensor is similar than concatenating the tensors list

In [1]:
# %%time 

# from transformers import EsmTokenizer, EsmModel
# import torch
# from torch.utils.data import DataLoader
# from datasets import load_dataset

# # Parameters
# batch_size = 32
# seq_length = 20
# model_name = "facebook/esm2_t33_650M_UR50D"
# data_file = "../data/c_test_downs_exp1.csv"

# # Load dataset (no tokenization here)
# dataset = load_dataset("csv", data_files=data_file, split="train")

# # Load model and tokenizer
# tokenizer = EsmTokenizer.from_pretrained(model_name)
# model = EsmModel.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda")
# model.eval()  # Set the model to evaluation mode

# # Define a collate function for on-the-fly batch tokenization
# def collate_fn(batch):
#     # Extract raw sequences from each example
#     sequences = [item['cleavage_window'] for item in batch]
#     # Tokenize the batch with padding to max_length for uniformity
#     batch_encoding = tokenizer(sequences, padding="max_length", truncation=True, max_length=seq_length, return_tensors="pt")
#     return batch_encoding.input_ids, batch_encoding.attention_mask

# # Create DataLoader with the collate_fn for dynamic tokenization
# dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn,num_workers=6)

# # Process batches through the model
# result_embeddings = []
# with torch.no_grad():
#     for input_ids, attention_mask in dataloader:
#         input_ids = input_ids.to("cuda")
#         attention_mask = attention_mask.to("cuda")
        
#         # Forward pass to get embeddings
#         outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
#         # Collect the embeddings
#         result_embeddings.append(outputs.last_hidden_state)

# # Concatenate all embeddings into a single tensor
# all_embeddings = torch.cat(result_embeddings, dim=0)



Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 1min 7s, sys: 8 s, total: 1min 15s
Wall time: 1min 14s


### protT5 embeddings

In [14]:
model_name = ProtT5[0]

t5_avg_dict = embed_data(model_name, downsampled_dataset, emb_type = "avg")
t5_concat_dict = embed_data(model_name, downsampled_dataset, emb_type = "concat")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/50540 [00:00<?, ? examples/s]

50540it [02:07, 395.59it/s]                                                                                             
50540it [02:05, 404.03it/s]                                                                                             


In [17]:
with open(outdir/"t5_avg_test50540.pkl", "wb") as f:
    pickle.dump(t5_avg_dict, f)

with open(outdir/"t5_concat_test50540.pkl", "wb") as f:
    pickle.dump(t5_concat_dict, f)